# Liu2024 — S-JEPA Embedding Probe + Filter-Bank Feature Fusion

**Goal.** Test whether frozen S-JEPA PreLocal embeddings carry left/right MI information when paired
with a small, stable classical classifier (shrinkage LDA or L2-logistic), and whether adding
**filter-bank Riemannian tangent features** *on top* (feature-level fusion) helps.

**Three feature sets** (`CONFIG["feature_set"]`): `sjepa` | `riemann` | `sjepa+riemann`.

**Leakage discipline (strict).** Per fold, everything data-driven — spatial_conv fine-tune (if enabled),
covariance tangent reference, PCA, and the classifier — is fit on the **train fold only**. Band-pass is a
fixed transform (safe pre-split); per-trial z-scoring is fine. The test fold is never seen during fitting.

> Logging mirrors `liu2024_source_mat_sjepa_prelocal_augmented`: every `print` is timestamped and tee'd to
> `run.log`; a `RUN_ID` is hashed from CONFIG; `config.json` and the CSV/JSON artifact set are written to a
> per-run artifact dir. **Edit the single CONFIG cell (or apply a sweep JSON) and re-run.**

# 1. Imports

In [1]:
import os, re, sys, json, time, random, hashlib, builtins, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy import signal as sp_signal

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim

import mne
mne.set_log_level("WARNING")
from braindecode.models import SignalJEPA_PreLocal

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

def resolve_device(cfg_device="auto"):
    if cfg_device != "auto":
        return torch.device(cfg_device)
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

print(f"torch {torch.__version__} | numpy {np.__version__}")

/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.10.0 | numpy 2.4.3


# 2. CONFIG  *(edit this one cell, or apply a sweep JSON, then re-run)*

In [2]:
CONFIG = {
    # ---- identity (shows up in logs + artifact names) ----
    "experiment_name": "A_probe_sjepa_lda_frozen",
    "config_note":     "frozen S-JEPA mean-pooled embeddings + shrinkage LDA",

    # ---- paths ----
    "data_root":    "../../liu2024_data/liu2024_figshare/sourcedata",
    "artifact_dir": "../../artifacts/liu2024_sjepa_embed_probe_fusion",

    # ---- S-JEPA checkpoint ----
    "sjepa_repo_id":         "braindecode/signal-jepa_without-chans",
    "sjepa_checkpoint_path": None,      # set to a local .pt to use your Liu2024-pretrained backbone
    "sjepa_pretrained":      True,      # False = random-init control

    # ---- dataset ----
    "subjects":   "all",
    "random_state": 2026,
    "seed":         2026,
    "sfreq_raw":    500,
    "sfreq_model":  128,
    # MI cue ~2.0 s into the 8 s trial; this 4.195 s model window runs ~1.5-5.7 s.
    "sjepa_mi_window_s":   [1.5, 5.7],
    "sjepa_window_samples": 537,        # 4.195 s @ 128 Hz (PreLocal expectation)
    "sjepa_bandpass_hz":   [0.5, 40.0],

    # ---- cross-validation ----
    "cv_scheme": "sjepa_5fold",          # 'sjepa_5fold' | 'liu_repeated_holdout'
    "n_splits":  5,                      # sjepa_5fold -> 32 train / 8 test
    "n_repeats": 10,                     # liu_repeated_holdout only
    "test_size": 0.40,                   # liu_repeated_holdout only

    # ---- which feature set to classify ----
    "feature_set": "sjepa",              # 'sjepa' | 'riemann' | 'sjepa+riemann'

    # ---- S-JEPA embedding branch ----
    "embedding_hook":  "feature_encoder", # 'feature_encoder' (rich tokens) | 'spatial_conv'
    "embedding_pool":  "mean",            # 'mean'|'max'|'meanmax'|'flatten'
    "use_pca":         True,
    "pca_max_components": 15,             # clipped to (n_train-1)
    "finetune_spatial_conv": False,       # False = frozen (deterministic, recommended for fusion)
    "finetune_epochs":   30,
    "finetune_lr":       1e-3,
    "finetune_batch_size": 8,
    "finetune_patience": 10,

    # ---- Riemannian (filter-bank) branch ----
    # Canonical bands on the 128 Hz model window (full window). The FULL 7x19 honest TWFB
    # grid lives in the hybrid notebook; here we keep a compact, fast set.
    "riemann_bands":    [[8, 13], [13, 30], [8, 30]],   # mu, beta, broad
    "riemann_cov_estimator": "oas",       # 'scm'|'oas'|'lwf'
    "riemann_metric":   "riemann",
    "riemann_filter_order": 4,
    "riemann_pca_max_components": 15,

    # ---- classifier on the (possibly fused) feature vector ----
    "classifier": "shrinkage_lda",        # 'shrinkage_lda' | 'logistic_l2'
    "logistic_C": 1.0,

    # ---- misc ----
    "device": "auto",
}

# ---- derived constants (do not edit; driven by CONFIG above) ----
DATA_ROOT   = Path(CONFIG["data_root"])
WINDOW_SAMPLES = CONFIG["sjepa_window_samples"]
SFREQ_RAW   = CONFIG["sfreq_raw"]
SFREQ_MODEL = CONFIG["sfreq_model"]
print(f"feature_set={CONFIG['feature_set']} | clf={CONFIG['classifier']} | cv={CONFIG['cv_scheme']} | finetune={CONFIG['finetune_spatial_conv']}")

feature_set=sjepa | clf=shrinkage_lda | cv=sjepa_5fold | finetune=False


## 2.1 Logging & Artifact Init

In [3]:
# --- Run ID, artifact dir, and timestamped logging tee'd to run.log (mirrors prelocal_augmented) ---
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_hash = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write(stream, text):
    try:
        stream.write(text)
    except UnicodeEncodeError:
        enc = getattr(stream, "encoding", None) or "utf-8"
        stream.write(text.encode(enc, errors="replace").decode(enc, errors="replace"))

_ORIG_PRINT = builtins.print
def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " "); end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False); file = kwargs.pop("file", None)
    msg = sep.join(str(a) for a in args)
    lead = len(msg) - len(msg.lstrip("\n")); body = msg[lead:]
    def w(t):
        _safe_write(sys.stdout if file is None else file, t)
    if lead:
        w("\n" * lead); _safe_write(_LOG_FILE_HANDLE, "\n" * lead)
    if body:
        stamped = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {body}"
        w(stamped + end); _safe_write(_LOG_FILE_HANDLE, stamped + end)
    else:
        w(end); _safe_write(_LOG_FILE_HANDLE, end)
    if flush:
        sys.stdout.flush(); _LOG_FILE_HANDLE.flush()
builtins.print = _timestamped_print

# Reproducibility
SEED = int(CONFIG.get("seed", CONFIG.get("random_state", 2026)))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = resolve_device(CONFIG.get("device", "auto"))
with open(ARTIFACT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

print("=" * 70)
print(f"Experiment: {CONFIG.get('experiment_name')}")
print(f"Note:       {CONFIG.get('config_note')}")
print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Device:     {DEVICE} | seed={SEED}")
print("=" * 70)

[2026-06-17 00:01:55] ======================================================================
[2026-06-17 00:01:55] Experiment: A_probe_sjepa_lda_frozen
[2026-06-17 00:01:55] Note:       frozen S-JEPA mean-pooled embeddings + shrinkage LDA
[2026-06-17 00:01:55] Run ID:     20260617_0001_9f244aad
[2026-06-17 00:01:55] Artifacts:  ../../artifacts/liu2024_sjepa_embed_probe_fusion/20260617_0001_9f244aad
[2026-06-17 00:01:55] Device:     mps | seed=2026
[2026-06-17 00:01:55] ======================================================================


# 3. Liu2024 Channel Constants

In [4]:
SOURCE_EEG_NAMES_30 = [
    "Fp1","Fp2","Fz","F3","F4","F7","F8","FCz","FC3","FC4",
    "FT7","FT8","Cz","C3","C4","T3","T4","CPz",
    "CP3","CP4","TP7","TP8","Pz","P3","P4","T5","T6","Oz","O1","O2",
]
CPZ_IDX      = 17                       # CPz is the source reference -> dropped
EEG_KEEP_IDX = [i for i in range(30) if i != CPZ_IDX]
EEG_NAMES    = [SOURCE_EEG_NAMES_30[i] for i in EEG_KEEP_IDX]
N_CHANS      = len(EEG_KEEP_IDX)        # 29
MARKER_SOURCE_COL = 32                  # 0-based col of marker channel in the 33-col source (==2 at MI onset)

def make_liu_info(sfreq):
    info = mne.create_info(ch_names=EEG_NAMES, sfreq=float(sfreq), ch_types=["eeg"] * N_CHANS)
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")
    return info

MNE_INFO = make_liu_info(CONFIG["sfreq_model"])
CHS_INFO = MNE_INFO["chs"]
print(f"N channels: {N_CHANS} | first 5: {EEG_NAMES[:5]}")

[2026-06-17 00:01:55] N channels: 29 | first 5: ['Fp1', 'Fp2', 'Fz', 'F3', 'F4']


# 4. Data Loading & Preprocessing (S-JEPA window)

In [5]:
def subject_id_from_path(path):
    m = re.search(r"sub[-_ ]?(\d{1,2})", str(path), flags=re.IGNORECASE)
    return int(m.group(1)) if m else int(re.findall(r"\d+", Path(path).stem)[-1])

def _load_mat_arrays(mat_path):
    """Return raw_data (40,33,4000) float64, y (40,) in {0,1}."""
    from scipy.io import loadmat
    mat = loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)
    raw = mat.get("rawdata", mat.get("data", None)); lab = mat.get("labels", mat.get("label", None))
    if raw is None or lab is None:
        for k, v in mat.items():
            if k.startswith("__"): continue
            if hasattr(v, "_fieldnames"):
                if raw is None and "rawdata" in v._fieldnames: raw = getattr(v, "rawdata")
                if lab is None and "label"   in v._fieldnames: lab = getattr(v, "label")
            elif raw is None and isinstance(v, np.ndarray) and v.ndim == 3:
                raw = v
    raw = np.asarray(raw, dtype=np.float64)
    trial_ax = next(ax for ax, sz in enumerate(raw.shape) if sz == 40)
    raw = np.moveaxis(raw, trial_ax, 0)
    if raw.shape[1] != 33 and raw.shape[2] == 33:
        raw = raw.transpose(0, 2, 1)
    assert raw.shape == (40, 33, 4000), f"shape {raw.shape}"
    y = np.asarray(lab, dtype=int).ravel()
    if set(np.unique(y).tolist()).issubset({1, 2}): y = y - 1
    assert len(y) == 40 and set(np.unique(y).tolist()).issubset({0, 1}), f"labels {np.unique(y)}"
    return raw, y

def preprocess_sjepa(mat_path, cfg):
    """Liu .mat -> (40, 29, window_samples) at sfreq_model, y. MNE avg-ref + resample + bandpass."""
    raw, y = _load_mat_arrays(mat_path)
    eeg = raw[:, EEG_KEEP_IDX, :].astype(np.float64)               # 40 x 29 x 4000
    n_trials, n_ch, n_t = eeg.shape
    fs_raw, fs_mod = cfg["sfreq_raw"], cfg["sfreq_model"]
    cont = eeg.transpose(1, 0, 2).reshape(n_ch, n_trials * n_t) * 1e-6   # channels x (trials*time), Volts
    info = mne.create_info(ch_names=EEG_NAMES, sfreq=float(fs_raw), ch_types=["eeg"] * N_CHANS)
    rm = mne.io.RawArray(cont, info, verbose=False)
    rm.set_eeg_reference("average", projection=False, verbose=False)
    rm.resample(fs_mod, npad="auto", verbose=False)
    bp_lo, bp_hi = cfg["sjepa_bandpass_hz"]
    rm.filter(bp_lo, bp_hi, method="fir", phase="zero", verbose=False)
    data = rm.get_data() * 1e6
    n_tr = int(n_t * fs_mod / fs_raw)
    data = data.reshape(N_CHANS, n_trials, n_tr).transpose(1, 0, 2)      # 40 x 29 x n_tr
    s0 = int(cfg["sjepa_mi_window_s"][0] * fs_mod); s1 = s0 + cfg["sjepa_window_samples"]
    assert s1 <= n_tr, f"window [{s0}:{s1}] > {n_tr}"
    X = data[:, :, s0:s1].astype(np.float32)
    assert X.shape == (40, N_CHANS, cfg["sjepa_window_samples"]) and np.isfinite(X).all()
    return X, y

def find_mat_files(root):
    root = Path(root)
    if not root.exists(): raise FileNotFoundError(f"data root not found: {root}")
    return sorted(root.rglob("*.mat"))

# 5. Filter-bank Riemannian features (for fusion)

In [6]:
# --- Filter-bank Riemannian tangent features on the 128 Hz model window ---
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace

def _bandpass_128(X, lo, hi, fs, order):
    ny = 0.5 * fs
    b, a = sp_signal.butter(order, [lo / ny, hi / ny], btype="band")
    return sp_signal.filtfilt(b, a, X, axis=-1)   # zero-phase

def riemann_tangent_features(X_window, cfg, tr_idx, te_idx):
    """
    X_window: (n_trials, 29, window_samples) at sfreq_model.
    For each canonical band: band-pass -> per-trial covariance -> TangentSpace (fit on TRAIN) -> concat.
    Returns (feat_train, feat_test). Leakage-safe: TangentSpace reference fit on train covariances only.
    """
    fs = cfg["sfreq_model"]; order = cfg["riemann_filter_order"]
    feats_tr, feats_te = [], []
    for (lo, hi) in cfg["riemann_bands"]:
        Xb = _bandpass_128(X_window, lo, hi, fs, order).astype(np.float64)
        covs = Covariances(estimator=cfg["riemann_cov_estimator"]).transform(Xb)   # (n,29,29)
        ts = TangentSpace(metric=cfg["riemann_metric"]).fit(covs[tr_idx])
        Z = ts.transform(covs)                                                     # (n, 435)
        feats_tr.append(Z[tr_idx]); feats_te.append(Z[te_idx])
    return np.concatenate(feats_tr, axis=1), np.concatenate(feats_te, axis=1)

def _pca_fit_transform(F_tr, F_te, max_comp, seed):
    if max_comp is None or F_tr.shape[1] <= max_comp:
        return F_tr, F_te, int(F_tr.shape[1]), float("nan")
    n = min(max_comp, F_tr.shape[0] - 1)
    pca = PCA(n_components=n, random_state=seed).fit(F_tr)
    return pca.transform(F_tr), pca.transform(F_te), int(n), float(np.sum(pca.explained_variance_ratio_))

print("Riemannian feature builder defined.")

[2026-06-17 00:01:55] Riemannian feature builder defined.


# 6. S-JEPA Model + Embedding Extraction

In [7]:
NEW_LAYER_PREFIXES = ("spatial_conv.", "final_layer.")

def load_sjepa_model(cfg, n_chans, chs_info, n_times, n_outputs=2):
    """Load SignalJEPA_PreLocal (pretrained or random) and freeze all but spatial_conv + final_layer."""
    kwargs = dict(n_chans=n_chans, chs_info=chs_info, n_times=n_times, n_outputs=n_outputs)
    if not cfg.get("sjepa_pretrained", True):
        print("S-JEPA: RANDOM init (control) -> no pretrained weights loaded")
        model = SignalJEPA_PreLocal(**kwargs)
    elif cfg.get("sjepa_checkpoint_path"):
        print(f"S-JEPA: loading local checkpoint {cfg['sjepa_checkpoint_path']}")
        model = SignalJEPA_PreLocal(**kwargs)
        state = torch.load(cfg["sjepa_checkpoint_path"], map_location="cpu")
        miss, unexp = model.load_state_dict(state, strict=False)
        print(f"  missing={len(miss)} unexpected={len(unexp)}")
    else:
        print(f"S-JEPA: from_pretrained {cfg['sjepa_repo_id']}")
        model = SignalJEPA_PreLocal.from_pretrained(cfg["sjepa_repo_id"], **kwargs, strict=False)
    for p in model.parameters():
        p.requires_grad = False
    for name, p in model.named_parameters():
        if any(name.startswith(pf) for pf in NEW_LAYER_PREFIXES):
            p.requires_grad = True
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  params total={tot:,} trainable={tr:,} (spatial_conv+final_layer)")
    return model

In [8]:
import copy

class TrialDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(np.asarray(X, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y, dtype=np.int64))
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def finetune_spatial_conv(model, X_train, y_train, cfg, device):
    """Fine-tune spatial_conv + final_layer on the train fold only (new-pre-local spirit)."""
    model = copy.deepcopy(model)
    for name, module in model.named_modules():
        if any(name.startswith(pf.rstrip(".")) for pf in NEW_LAYER_PREFIXES):
            if hasattr(module, "reset_parameters"):
                module.reset_parameters()
    model.train()
    opt = optim.Adam([p for p in model.parameters() if p.requires_grad],
                     lr=cfg["finetune_lr"], weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    n = len(X_train); n_val = max(2, int(0.2 * n))
    idx = np.random.permutation(n); vi, ti = idx[:n_val], idx[n_val:]
    dl_tr = torch.utils.data.DataLoader(TrialDataset(X_train[ti], y_train[ti]),
                                        batch_size=cfg["finetune_batch_size"], shuffle=True)
    dl_va = torch.utils.data.DataLoader(TrialDataset(X_train[vi], y_train[vi]),
                                        batch_size=cfg["finetune_batch_size"], shuffle=False)
    best = float("inf"); best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()}); pc = 0
    for _ in range(cfg["finetune_epochs"]):
        model.train()
        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
        model.eval(); vl = 0.0
        with torch.no_grad():
            for xb, yb in dl_va:
                vl += crit(model(xb.to(device)), yb.to(device)).item()
        vl /= max(len(dl_va), 1)
        if vl < best - 1e-4:
            best = vl; best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()}); pc = 0
        else:
            pc += 1
            if pc >= cfg["finetune_patience"]:
                break
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    return model

@torch.no_grad()
def extract_embeddings(model, X, cfg, device):
    """Per-trial rich embedding via forward hook on an intermediate module (NOT the 2-D logits)."""
    model.eval()
    pool = cfg.get("embedding_pool", "mean"); hook_name = cfg.get("embedding_hook", "feature_encoder")
    target = getattr(model, hook_name, None)
    if target is None:
        raise AttributeError(f"no submodule '{hook_name}' to hook")
    cap = {}
    def _hook(m, i, o): cap["z"] = o.detach()
    h = target.register_forward_hook(_hook)
    dl = torch.utils.data.DataLoader(TrialDataset(X, np.zeros(len(X), dtype=np.int64)), batch_size=32, shuffle=False)
    embs = []
    try:
        for xb, _ in dl:
            _ = model(xb.to(device)); z = cap["z"]
            if z.dim() == 2:
                feat = z
            else:
                axis = 2 if hook_name == "spatial_conv" else 1
                if pool == "flatten":   feat = z.flatten(start_dim=1)
                elif pool == "max":     feat = z.max(dim=axis).values.flatten(start_dim=1)
                elif pool == "meanmax": feat = torch.cat([z.mean(dim=axis), z.max(dim=axis).values], dim=-1).flatten(start_dim=1)
                else:                   feat = z.mean(dim=axis).flatten(start_dim=1)
            embs.append(feat.cpu().numpy())
    finally:
        h.remove()
    out = np.concatenate(embs, axis=0).astype(np.float32)
    if not hasattr(extract_embeddings, "_printed"):
        print(f"  [hook={hook_name} pool={pool}] embedding matrix: {out.shape}"); extract_embeddings._printed = True
    return out

# 7. Classifier + Diagnostics

In [9]:
def make_clf(cfg, which="classifier"):
    name = cfg.get(which, "shrinkage_lda")
    if name == "shrinkage_lda":
        return LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    if name == "logistic_l2":
        return LogisticRegression(C=cfg.get("logistic_C", 1.0), penalty="l2", solver="lbfgs", max_iter=500)
    raise ValueError(f"unknown classifier {name}")

def collapse_diagnostics(y_pred, n_classes=2):
    counts = np.bincount(np.asarray(y_pred, dtype=int), minlength=n_classes)
    dom = counts.max() / counts.sum() if counts.sum() > 0 else 1.0
    return {"collapse_flag": bool(dom > 0.95), "collapse_ratio": float(dom), "pred_counts": counts.tolist()}

# 8. Feature Assembly + Per-Subject CV Runner

In [10]:
def assemble_features(X_window, y, cfg, tr_idx, te_idx, base_model, device):
    """Build the per-fold feature matrices for the requested feature_set, PCA each block on train only, concat."""
    blocks_tr, blocks_te, meta = [], [], {}
    fs_set = cfg["feature_set"]

    if fs_set in ("sjepa", "sjepa+riemann"):
        if cfg["finetune_spatial_conv"]:
            model = finetune_spatial_conv(base_model, X_window[tr_idx], y[tr_idx], cfg, device)
        else:
            model = copy.deepcopy(base_model); model.eval()
        emb = extract_embeddings(model, X_window, cfg, device)          # (n, D) for ALL trials
        e_tr, e_te = emb[tr_idx], emb[te_idx]
        e_tr, e_te, npc, var = _pca_fit_transform(e_tr, e_te, cfg["pca_max_components"], cfg["random_state"])
        blocks_tr.append(e_tr); blocks_te.append(e_te)
        meta["sjepa_dim"] = int(emb.shape[1]); meta["sjepa_pca"] = npc; meta["sjepa_pca_var"] = var

    if fs_set in ("riemann", "sjepa+riemann"):
        r_tr, r_te = riemann_tangent_features(X_window, cfg, tr_idx, te_idx)
        r_tr, r_te, npc, var = _pca_fit_transform(r_tr, r_te, cfg["riemann_pca_max_components"], cfg["random_state"])
        blocks_tr.append(r_tr); blocks_te.append(r_te)
        meta["riemann_dim_pca"] = npc; meta["riemann_pca_var"] = var

    F_tr = np.concatenate(blocks_tr, axis=1); F_te = np.concatenate(blocks_te, axis=1)
    meta["fused_dim"] = int(F_tr.shape[1])
    return F_tr, F_te, meta

def make_subject_splits(X, y, cfg):
    scheme = cfg.get("cv_scheme", "sjepa_5fold")
    if scheme == "sjepa_5fold":
        sp = StratifiedKFold(n_splits=cfg["n_splits"], shuffle=True, random_state=cfg["random_state"])
    elif scheme == "liu_repeated_holdout":
        sp = StratifiedShuffleSplit(n_splits=cfg["n_repeats"], test_size=cfg["test_size"], random_state=cfg["random_state"])
    else:
        raise ValueError(f"unknown cv_scheme {scheme}")
    for k, (tr, te) in enumerate(sp.split(X, y)):
        yield k, tr, te

def run_subject(sid, X, y, cfg, base_model, device):
    rows = []
    for fold_idx, tr, te in make_subject_splits(X, y, cfg):
        y_tr, y_te = y[tr], y[te]
        assert len(np.unique(y_tr)) == 2 and len(np.unique(y_te)) == 2
        meta = {}
        try:
            F_tr, F_te, meta = assemble_features(X, y, cfg, tr, te, base_model, device)
            clf = make_clf(cfg); clf.fit(F_tr, y_tr)
            tr_pred = clf.predict(F_tr); y_pred = clf.predict(F_te)
        except Exception as exc:
            print(f"  Sub {sid} fold {fold_idx} ERROR: {exc}")
            tr_pred = np.zeros(len(y_tr), int); y_pred = np.zeros(len(y_te), int)
        cm = confusion_matrix(y_te, y_pred, labels=[0, 1]); cma = np.array(cm)
        diag = collapse_diagnostics(y_pred)
        rows.append({
            "subject_id": int(sid), "fold_id": int(fold_idx), "feature_set": cfg["feature_set"],
            "cv_scheme": cfg.get("cv_scheme"), "classifier": cfg["classifier"],
            "accuracy": float(accuracy_score(y_te, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_te, y_pred)),
            "train_accuracy": float(accuracy_score(y_tr, tr_pred)),
            "train_balanced_accuracy": float(balanced_accuracy_score(y_tr, tr_pred)),
            "left_recall": float(cma[0, 0] / cma[0].sum()) if cma[0].sum() else float("nan"),
            "right_recall": float(cma[1, 1] / cma[1].sum()) if cma[1].sum() else float("nan"),
            "confusion_matrix": cm.tolist(),
            "collapse_flag": diag["collapse_flag"], "collapse_ratio": diag["collapse_ratio"],
            "pred_counts": diag["pred_counts"],
            "y_train_counts": np.bincount(y_tr, minlength=2).tolist(),
            "y_test_counts":  np.bincount(y_te, minlength=2).tolist(),
            "n_train": int(len(y_tr)), "n_test": int(len(y_te)),
            **{f"meta_{k}": v for k, v in meta.items()},
        })
    return rows

print("Feature assembly + subject runner defined.")

[2026-06-17 00:01:56] Feature assembly + subject runner defined.


# 9. Run All Subjects

In [11]:
mat_files  = find_mat_files(DATA_ROOT)
all_sids   = sorted({subject_id_from_path(f) for f in mat_files})
SUBJECT_IDS = all_sids if CONFIG["subjects"] == "all" else sorted(int(s) for s in CONFIG["subjects"])
sid_to_path = {subject_id_from_path(f): f for f in mat_files if subject_id_from_path(f) in SUBJECT_IDS}
print(f"Found {len(mat_files)} .mat files | using {len(SUBJECT_IDS)} subjects")

# Build the S-JEPA base model once (only needed for sjepa feature sets)
BASE_MODEL = None
if CONFIG["feature_set"] in ("sjepa", "sjepa+riemann"):
    BASE_MODEL = load_sjepa_model(CONFIG, N_CHANS, CHS_INFO, WINDOW_SAMPLES).to(DEVICE)

ALL_FOLD_RESULTS, SUBJECT_SUMMARIES = [], []
print("=" * 70)
for sid in SUBJECT_IDS:
    mat_path = sid_to_path.get(sid)
    if mat_path is None:
        print(f"  Sub {sid:02d}: no file, skipping"); continue
    try:
        X, y = preprocess_sjepa(mat_path, CONFIG)
    except Exception as exc:
        print(f"  Sub {sid:02d}: preprocess error — {exc}"); continue
    rows = run_subject(sid, X, y, CONFIG, BASE_MODEL, DEVICE)
    ALL_FOLD_RESULTS.extend(rows)
    baccs = [r["balanced_accuracy"] for r in rows]; coll = sum(r["collapse_flag"] for r in rows)
    SUBJECT_SUMMARIES.append({
        "subject_id": sid, "mean_accuracy": float(np.mean([r["accuracy"] for r in rows])),
        "mean_balanced_accuracy": float(np.mean(baccs)), "std_balanced_accuracy": float(np.std(baccs)),
        "n_folds": len(rows), "n_collapsed_folds": coll,
    })
    print(f"  Sub {sid:02d}: bal_acc={np.mean(baccs)*100:5.1f}% ± {np.std(baccs)*100:4.1f}%  collapse={coll}/{len(rows)}")
print("=" * 70)
print(f"Done. Total folds: {len(ALL_FOLD_RESULTS)}")

[2026-06-17 00:01:56] Found 50 .mat files | using 50 subjects
[2026-06-17 00:01:56] S-JEPA: from_pretrained braindecode/signal-jepa_without-chans


[2026-06-17 00:01:56]   params total=16,010 trainable=2,170 (spatial_conv+final_layer)
[2026-06-17 00:01:56] ======================================================================
[2026-06-17 00:01:58]   [hook=feature_encoder pool=mean] embedding matrix: (40, 64)
[2026-06-17 00:01:58]   Sub 01: bal_acc= 65.0% ± 14.6%  collapse=0/5
[2026-06-17 00:01:58]   Sub 02: bal_acc= 55.0% ± 10.0%  collapse=0/5
[2026-06-17 00:01:59]   Sub 03: bal_acc= 57.5% ± 17.0%  collapse=0/5
[2026-06-17 00:01:59]   Sub 04: bal_acc= 62.5% ± 17.7%  collapse=0/5
[2026-06-17 00:01:59]   Sub 05: bal_acc= 52.5% ± 14.6%  collapse=0/5
[2026-06-17 00:02:00]   Sub 06: bal_acc= 52.5% ±  9.4%  collapse=0/5
[2026-06-17 00:02:00]   Sub 07: bal_acc= 80.0% ± 12.7%  collapse=0/5
[2026-06-17 00:02:00]   Sub 08: bal_acc= 40.0% ± 25.5%  collapse=0/5
[2026-06-17 00:02:01]   Sub 09: bal_acc= 45.0% ± 17.0%  collapse=0/5
[2026-06-17 00:02:01]   Sub 10: bal_acc= 70.0% ± 25.7%  collapse=0/5
[2026-06-17 00:02:02]   Sub 11: bal_acc= 50.0%

# 10. Aggregate, Save Artifacts, Plots

In [12]:
# --- Aggregate, save artifacts (mirrors prelocal_augmented naming), plots ---
if not ALL_FOLD_RESULTS:
    raise RuntimeError("No folds completed — check data_root and that the run cell executed.")

fold_df    = pd.DataFrame(ALL_FOLD_RESULTS)
subject_df = pd.DataFrame(SUBJECT_SUMMARIES)
cm_total = np.zeros((2, 2), dtype=int)
for r in ALL_FOLD_RESULTS:
    cm_total += np.array(r["confusion_matrix"])
lr = cm_total[0, 0] / cm_total[0].sum() if cm_total[0].sum() else float("nan")
rr = cm_total[1, 1] / cm_total[1].sum() if cm_total[1].sum() else float("nan")

GLOBAL_METRICS = {
    "experiment_name": CONFIG["experiment_name"], "feature_set": CONFIG["feature_set"],
    "classifier": CONFIG["classifier"], "cv_scheme": CONFIG.get("cv_scheme"),
    "sjepa_pretrained": CONFIG.get("sjepa_pretrained"), "finetune_spatial_conv": CONFIG.get("finetune_spatial_conv"),
    "n_subjects": len(SUBJECT_SUMMARIES), "n_folds_total": len(ALL_FOLD_RESULTS),
    "mean_accuracy": float(fold_df["accuracy"].mean()), "std_accuracy": float(fold_df["accuracy"].std()),
    "mean_balanced_accuracy": float(fold_df["balanced_accuracy"].mean()),
    "std_balanced_accuracy": float(fold_df["balanced_accuracy"].std()),
    "n_collapsed_folds": int(fold_df["collapse_flag"].sum()),
    "collapse_rate": float(fold_df["collapse_flag"].mean()),
    "left_recall_agg": float(lr), "right_recall_agg": float(rr),
    "confusion_matrix": cm_total.tolist(),
}

fold_df.to_csv(ARTIFACT_DIR / "fold_level_results.csv", index=False)
subject_df.to_csv(ARTIFACT_DIR / "subject_level_summary.csv", index=False)
with open(ARTIFACT_DIR / "global_metrics.json", "w") as f: json.dump(GLOBAL_METRICS, f, indent=2)
with open(ARTIFACT_DIR / "cv_results.json", "w") as f: json.dump(ALL_FOLD_RESULTS, f, indent=2, default=str)
with open(ARTIFACT_DIR / "subject_metrics.json", "w") as f: json.dump(SUBJECT_SUMMARIES, f, indent=2)
with open(ARTIFACT_DIR / "run_metadata.json", "w") as f:
    json.dump({"run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR), "config": CONFIG,
               "n_subjects": len(SUBJECT_SUMMARIES), "n_folds": len(ALL_FOLD_RESULTS)}, f, indent=2, default=str)

print("=" * 70)
print(f"GLOBAL — S-JEPA probe/fusion [{CONFIG['feature_set']} | {CONFIG['classifier']}]")
print(f"  Balanced accuracy: {GLOBAL_METRICS['mean_balanced_accuracy']*100:.2f}% ± {GLOBAL_METRICS['std_balanced_accuracy']*100:.2f}%")
print(f"  Accuracy:          {GLOBAL_METRICS['mean_accuracy']*100:.2f}% ± {GLOBAL_METRICS['std_accuracy']*100:.2f}%")
print(f"  Left/Right recall: {lr*100:.1f}% / {rr*100:.1f}%")
print(f"  Collapsed folds:   {GLOBAL_METRICS['n_collapsed_folds']}/{len(ALL_FOLD_RESULTS)} ({GLOBAL_METRICS['collapse_rate']*100:.1f}%)")
print(f"  Artifacts: {ARTIFACT_DIR}")
print("=" * 70)

# Plots
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(cm_total, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred L", "Pred R"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["True L", "True R"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm_total[i, j]), ha="center", va="center")
ax.set_title(f"CM — {CONFIG['feature_set']} / {CONFIG['classifier']}")
plt.colorbar(im, ax=ax); plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "confusion_matrix.png", dpi=200, bbox_inches="tight"); plt.close(fig)

fig, ax = plt.subplots(figsize=(max(8, len(subject_df) * 0.4), 4))
sids = subject_df["subject_id"].values; b = subject_df["mean_balanced_accuracy"].values * 100
ax.bar(sids, b, yerr=subject_df["std_balanced_accuracy"].values * 100, capsize=3, color="mediumpurple", alpha=0.85)
ax.axhline(50, color="red", ls="--", label="chance"); ax.axhline(b.mean(), color="orange", label=f"mean={b.mean():.1f}%")
ax.set_xlabel("subject"); ax.set_ylabel("balanced acc (%)"); ax.legend()
ax.set_xticks(sids); ax.set_xticklabels(sids, rotation=90, fontsize=7)
ax.set_title(f"Per-subject — {CONFIG['feature_set']} / {CONFIG['classifier']}")
plt.tight_layout(); plt.savefig(ARTIFACT_DIR / "subject_performance.png", dpi=200, bbox_inches="tight"); plt.close(fig)
print("Saved CSV/JSON artifacts + confusion_matrix.png + subject_performance.png")

[2026-06-17 00:02:16] ======================================================================
[2026-06-17 00:02:16] GLOBAL — S-JEPA probe/fusion [sjepa | shrinkage_lda]
[2026-06-17 00:02:16]   Balanced accuracy: 53.95% ± 17.25%
[2026-06-17 00:02:16]   Accuracy:          53.95% ± 17.25%
[2026-06-17 00:02:16]   Left/Right recall: 54.0% / 53.9%
[2026-06-17 00:02:16]   Collapsed folds:   5/250 (2.0%)
[2026-06-17 00:02:16]   Artifacts: ../../artifacts/liu2024_sjepa_embed_probe_fusion/20260617_0001_9f244aad
[2026-06-17 00:02:16] ======================================================================
[2026-06-17 00:02:16] Saved CSV/JSON artifacts + confusion_matrix.png + subject_performance.png
